# Long-context RAG — Anthropic-style contextual retrieval

Standard chunking strips a chunk from its surrounding context. The retriever then sees a passage with no idea which paper it came from or where in that paper it sits. **Contextual retrieval** (Anthropic, 2024) fixes this with one extra LLM call per chunk: it generates a short prefix that situates the chunk, prepends it, and embeds the augmented version.

The cost is N chunks × one cheap LLM call (cacheable). The gain reported by Anthropic is a ~35% drop in retrieval failure rate.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/11-long-context-rag'
SUBSET_DOC_IDS = ['synth-001', 'synth-002', 'synth-003', 'synth-004', 'synth-005']
DOCS = [d for d in load_corpus() if d.arxiv_id in SUBSET_DOC_IDS]
QA = [q for q in load_golden_qa() if any(s in SUBSET_DOC_IDS for s in q.source_ids)]
print('docs:', len(DOCS), '  Qs:', len(QA))

docs: 5   Qs: 15


## Split each abstract into two chunks

In [3]:
def split_doc(text):
    sentences = text.replace('?', '?.').replace('!', '!.').split('. ')
    mid = max(1, len(sentences) // 2)
    head = '. '.join(sentences[:mid]).strip()
    tail = '. '.join(sentences[mid:]).strip()
    if head and not head.endswith('.'): head += '.'
    if tail and not tail.endswith('.'): tail += '.'
    return [head, tail]

chunks = []  # list of (doc_id, chunk_idx, text)
for d in DOCS:
    for i, c in enumerate(split_doc(d.abstract)):
        chunks.append((d.arxiv_id, i, c))
print('total chunks:', len(chunks))

total chunks: 10


## Generate one context prefix per chunk

One small LLM call per chunk, but each is cheap and cacheable.

In [4]:
CTX_SYS = (
    'You are summarizing the position of a chunk inside its full document. Given the document '
    'and a single chunk, produce a SHORT (<=20 words) context prefix that situates the chunk. '
    'Do not restate the chunk. Output ONLY the prefix, no quoting, no extra prose.'
)
def prefix_for(doc, chunk):
    user = (
        f'<document>\n{doc.title}. {doc.abstract}\n</document>\n\n'
        f'<chunk>\n{chunk}\n</chunk>\n\nContext prefix:'
    )
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=CTX_SYS),
        Message(role='user', content=user),
    ]).content.strip()

doc_by_id = {d.arxiv_id: d for d in DOCS}
augmented = []
for did, ci, c in chunks:
    p = prefix_for(doc_by_id[did], c)
    augmented.append((did, ci, c, p))
for did, ci, c, p in augmented[:3]:
    print(f'{did}#{ci}  CTX: {p}')

synth-001#0  CTX: Opening of RA-MoE paper, motivating routing-aware sparse Mixture-of-Experts for latency-bounded inference.
synth-001#1  CTX: Continuation of RA-MoE paper: reports 38% p99 decode latency reduction at 47B parameters.
synth-002#0  CTX: Opening of HA-Retrieve paper, introducing hierarchical attention for million-token document retrieval.


## Compare retrieval recall — bare chunks vs contextualized chunks

In [5]:
bare_vecs = hash_embed([c for _, _, c, _ in augmented], dims=256, seed=0)
ctx_vecs  = hash_embed([f'{p} {c}' for _, _, c, p in augmented], dims=256, seed=0)

def recall_at_k(matrix, k=3):
    hits = 0
    for q in QA:
        qv = hash_embed([q.question], dims=256, seed=0)[0]
        idx, _ = cosine_topk(qv, matrix, k=k)
        retrieved_docs = {augmented[i][0] for i in idx}
        if retrieved_docs & set(q.source_ids): hits += 1
    return round(hits / len(QA), 4)

for k in (1, 3, 5):
    print(f'recall@{k}  bare={recall_at_k(bare_vecs, k)}  contextual={recall_at_k(ctx_vecs, k)}')

recall@1  bare=0.6667  contextual=0.8
recall@3  bare=0.8667  contextual=0.8667
recall@5  bare=0.8667  contextual=1.0


## What you can extend

* Combine contextual prefixes with hybrid search (`03-hybrid-search/`) — Anthropic reports the two stack additively.
* Use prompt caching on the document portion of the prefix call — cuts cost ~10x for large docs.
* For million-token docs: hierarchical context (chapter prefix → section prefix → chunk prefix).